# module-modules-iter-isinstance-dispatch — worked example 2: Check whether a model contains any BatchNorm layer

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `module-modules-iter-isinstance-dispatch`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import matplotlib.pyplot as plt

## Concept

`model.modules()` lets you inspect every layer in a network without knowing the architecture in advance. A common use-case is checking whether a model contains any normalization layers — for example, before deciding whether to call `model.eval()` to freeze the running statistics. `isinstance` with a tuple of types handles multi-variant checks (BatchNorm1d, BatchNorm2d, BatchNorm3d) in one expression.

## Worked solution

**Step 1 — define the target types.** We want to detect any BatchNorm variant: `(nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)`. Putting them in a tuple lets a single `isinstance` call match all three.

**Step 2 — iterate and short-circuit.** We walk `model.modules()` and as soon as we see one matching layer we can return `True` immediately. Using Python's built-in `any()` with a generator expression is idiomatic and efficient: it short-circuits on the first match.

**Step 3 — return `False` if the loop finishes without a match.** `any()` already handles this: if the generator is exhausted without yielding `True`, `any()` returns `False`.

**Why this matters.** If `has_batchnorm` returns `True`, you must call `.eval()` before inference to freeze running stats, otherwise the model will update them on the evaluation data, introducing data leakage.

In [ ]:
import torch
import torch.nn as nn

def has_batchnorm(model: nn.Module) -> bool:
    """Return True if model contains ANY BatchNorm layer (1d, 2d, or 3d)."""
    bn_types = (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)
    return any(isinstance(m, bn_types) for m in model.modules())

# Test on two networks.
net_with_bn = nn.Sequential(
    nn.Conv2d(3, 16, 3, padding=1),
    nn.BatchNorm2d(16),
    nn.ReLU(),
    nn.AdaptiveAvgPool2d(1),
    nn.Flatten(),
    nn.Linear(16, 10),
)

net_without_bn = nn.Sequential(
    nn.Linear(32, 64),
    nn.ReLU(),
    nn.Linear(64, 10),
)

print(f"net_with_bn has BatchNorm: {has_batchnorm(net_with_bn)}")     # True
print(f"net_without_bn has BatchNorm: {has_batchnorm(net_without_bn)}")  # False